# S?IT Aspect Sentiment Quad Prediction ? Kaggle

This notebook clones the project source from GitHub, locates the QuADAspect JSONL files added through Kaggle **Add Input**, prepares Stanza parsers, trains the three S?IT stages, and evaluates exact-match quadruple F1. Enable **GPU** and **Internet** in Kaggle Notebook settings.

The default base model is a local Kaggle Model input. Confirm that its directory contains `config.json`; change `BASE_MODEL` in the config cell if Kaggle mounts it elsewhere. Training saves the LoRA adapter and evaluation outputs under `/kaggle/working/s2it_outputs/`.


In [ ]:
# ---- CONFIG ----
# Repository is cloned from GitHub into Kaggle's writable working directory.
GITHUB_REPO = "https://github.com/hotuyen21pt/s2it.git"
PROJECT_DIR = "/kaggle/working/s2it"

# Add https://www.kaggle.com/datasets/tuynnguynh22280103/quadaspect via Add Input.
# Kaggle mounts it at /kaggle/input/quadaspect. Leave DATA_DIR=None to locate
# train.jsonl, dev.jsonl and test.jsonl anywhere inside this dataset directory.
DATASET_INPUT_DIR = "/kaggle/input/quadaspect"
DATA_DIR = None  # optionally set to the subfolder containing all three JSONL files
OUTPUT_DIR = "/kaggle/working/s2it_outputs"

# Hugging Face model ID OR local Kaggle Input directory with model files.
# Online: paper checkpoint. Offline: attach full base-model files and set its path here.
BASE_MODEL = "/kaggle/input/models/dipamc77/qwen2.5-7b-instruct/transformers/default/1/Qwen2.5-7B-Instruct"
OFFLINE_MODE = False  # Kaggle Settings > Internet must also be enabled
HF_CACHE_DIR = "/kaggle/working/hf_cache"
STANZA_RESOURCES_INPUT_DIR = None  # optional local Stanza resources directory
STANZA_MODEL_DIR = None  # optional custom folder containing Stanza model files
STANZA_MODEL_LANGUAGE = "en"
STANZA_CACHE_DIR = "/kaggle/working/stanza_resources"
DOWNLOAD_STANZA_MODELS = True
EPOCHS = 5
BATCH_SIZE = 4
GRADIENT_ACCUMULATION_STEPS = 2
LEARNING_RATE = 5e-5
LORA_RANK = 32
MAX_LENGTH = 2048
DATA_PERCENT = 10  # Run 10% of each split for a full-pipeline trial; set to 100 for all data.

# Kaggle parser allowlist: English, Vietnamese, Simplified Chinese, Japanese, German, Thai, French.
PARSER_LANGUAGES = []  # populated from train/dev/test JSONL after language tagging
PARSER_LANGUAGE = "auto"

RUN_TRAINING = True
RUN_INFERENCE = True
MODEL_DIR = f"{OUTPUT_DIR}/final"
MAX_EVAL_RECORDS = None
MAX_NEW_TOKENS = 512
PARSER_USE_GPU = False

from pathlib import Path
import os
import sys

os.environ["HF_HOME"] = str(Path(HF_CACHE_DIR).expanduser())
os.environ["HF_HUB_OFFLINE"] = "0"
os.environ["TRANSFORMERS_OFFLINE"] = "0"
STANZA_RESOURCES_INPUT_DIR = str(Path(STANZA_RESOURCES_INPUT_DIR).expanduser()) if STANZA_RESOURCES_INPUT_DIR else None
STANZA_CACHE_DIR = str(Path(STANZA_CACHE_DIR).expanduser())
STANZA_RESOURCES_DIR = STANZA_RESOURCES_INPUT_DIR or STANZA_CACHE_DIR
os.environ["STANZA_RESOURCES_DIR"] = STANZA_RESOURCES_DIR
Path(HF_CACHE_DIR).mkdir(parents=True, exist_ok=True)
PROJECT_DIR = str(Path(PROJECT_DIR).expanduser()) if PROJECT_DIR else None
DATASET_INPUT_DIR = str(Path(DATASET_INPUT_DIR).expanduser())
DATA_DIR = str(Path(DATA_DIR).expanduser()) if DATA_DIR else None
OUTPUT_DIR = str(Path(OUTPUT_DIR).expanduser())
MODEL_DIR = str(Path(MODEL_DIR).expanduser())
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"GitHub repo: {GITHUB_REPO}\nProject: {PROJECT_DIR}\nDataset: {DATASET_INPUT_DIR}\nData: {DATA_DIR or 'auto-detect'}\nOutput: {OUTPUT_DIR}")

## 1. Clone GitHub, find the dataset, and install dependencies

This notebook clones the latest source from `https://github.com/hotuyen21pt/s2it.git` into `/kaggle/working/s2it`. Enable **Internet** in Kaggle Settings to clone the repo and install dependencies. Add [QuADAspect](https://www.kaggle.com/datasets/tuynnguynh22280103/quadaspect) through **Add Input**; the notebook searches for `train.jsonl`, `dev.jsonl`, and `test.jsonl` under `/kaggle/input/quadaspect`. If the dataset has a different layout, update `DATASET_INPUT_DIR` or set `DATA_DIR` directly in the config cell.

In [ ]:
from pathlib import Path
import subprocess

input_root = Path("/kaggle/input")
input_folders = sorted(str(path) for path in input_root.iterdir()) if input_root.exists() else []

def is_s2it_project(path):
    return (path / "pyproject.toml").is_file() and (path / "src" / "asqp_s2it").is_dir()

project_path = Path(PROJECT_DIR)
if project_path.exists() and (project_path / ".git").is_dir():
    print(f"Updating repository in {project_path} ...")
    subprocess.check_call(["git", "-C", str(project_path), "pull", "--ff-only", "origin", "master"])
elif not is_s2it_project(project_path):
    if project_path.exists() and any(project_path.iterdir()):
        raise FileExistsError(
            f"{project_path} exists but is not a Git clone of the S2IT repository. "
            "Choose another PROJECT_DIR or remove that folder, then rerun."
        )
    project_path.parent.mkdir(parents=True, exist_ok=True)
    print(f"Cloning {GITHUB_REPO} into {project_path} ...")
    subprocess.check_call(["git", "clone", "--depth", "1", GITHUB_REPO, str(project_path)])
if not is_s2it_project(project_path):
    raise FileNotFoundError(f"GitHub clone is missing pyproject.toml or src/asqp_s2it: {project_path}")
PROJECT_DIR = str(project_path)
print(f"Loaded project from GitHub: {PROJECT_DIR}")

if STANZA_RESOURCES_INPUT_DIR is None:
    bundled_stanza_dir = project_path / "stanza_resources_en_vi"
    if (bundled_stanza_dir / "resources.json").is_file():
        STANZA_RESOURCES_INPUT_DIR = str(bundled_stanza_dir)
if STANZA_RESOURCES_INPUT_DIR:
    stanza_resources_path = Path(STANZA_RESOURCES_INPUT_DIR).expanduser()
    if not (stanza_resources_path / "resources.json").is_file():
        raise FileNotFoundError(f"Stanza resources.json not found under: {stanza_resources_path}")
    STANZA_RESOURCES_DIR = str(stanza_resources_path)
else:
    STANZA_RESOURCES_DIR = STANZA_CACHE_DIR
os.environ["STANZA_RESOURCES_DIR"] = STANZA_RESOURCES_DIR
print(f"Stanza resources: {STANZA_RESOURCES_DIR}")
SRC_DIR = str(project_path / "src")
for import_path in (PROJECT_DIR, SRC_DIR):
    if import_path not in sys.path:
        sys.path.insert(0, import_path)

if DATA_DIR is None:
    dataset_path = Path(DATASET_INPUT_DIR)
    preferred_roots = [dataset_path] if dataset_path.is_dir() else []
    kaggle_datasets_root = input_root / "datasets"
    if kaggle_datasets_root.is_dir() and kaggle_datasets_root not in preferred_roots:
        preferred_roots.append(kaggle_datasets_root)
    if not preferred_roots:
        preferred_roots = [input_root]

    required_files = {"train.jsonl", "dev.jsonl", "test.jsonl"}
    data_candidates = []
    for search_root in preferred_roots:
        for current, dirs, files in os.walk(search_root):
            dirs[:] = [name for name in dirs if not name.startswith(".")]
            if required_files.issubset(files):
                data_candidates.append(Path(current))
        if data_candidates:
            break
    if len(data_candidates) != 1:
        raise FileNotFoundError(
            "Could not find exactly one folder containing train.jsonl, dev.jsonl and test.jsonl. "
            f"Searched: {preferred_roots}; matches: {data_candidates}; "
            f"available /kaggle/input entries: {input_folders}. "
            "Set DATA_DIR to the exact folder containing all three JSONL files."
        )
    DATA_DIR = str(data_candidates[0])
data_path = Path(DATA_DIR)
base_model_path = Path(BASE_MODEL).expanduser()
if base_model_path.is_absolute() and not base_model_path.is_dir():
    raise FileNotFoundError(
        f"Local base model path does not exist: {base_model_path}. "
        "Check the Kaggle Model mount path or set BASE_MODEL to a Hugging Face model ID."
    )
if base_model_path.is_dir() and not (base_model_path / "config.json").is_file():
    raise FileNotFoundError(f"The model directory has no config.json: {base_model_path}")
if OFFLINE_MODE and not base_model_path.is_dir():
    raise FileNotFoundError(
        f"Local base model not found: {base_model_path}. Attach the model as a Kaggle Input and set BASE_MODEL to its config.json folder."
    )
for split in ("train", "dev", "test"):
    assert (data_path / f"{split}.jsonl").is_file(), f"Missing {split}.jsonl in DATA_DIR={data_path}"
print(f"Found project and train/dev/test JSONL under {DATA_DIR}.")

import importlib.util
packages = {
    "stanza": ("stanza", "stanza>=1.8.2"),
    "langdetect": ("langdetect", "langdetect>=1.0.9"),
    "peft": ("peft", "peft>=0.12.0"),
    "transformers": ("transformers", "transformers>=4.45.0"),
    "accelerate": ("accelerate", "accelerate>=0.34.0"),
    "torch": ("torch", None),
    "pandas": ("pandas", "pandas>=2.0.0"),
}
missing = [spec for module, (name, spec) in packages.items() if importlib.util.find_spec(module) is None and spec]
if missing:
    print("Installing missing packages from PyPI:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
missing_modules = [module for module, (_, spec) in packages.items() if importlib.util.find_spec(module) is None]
if missing_modules:
    raise RuntimeError(f"Required packages still missing after installation: {missing_modules}")

from importlib.metadata import PackageNotFoundError, version
from packaging.version import Version

# Kaggle may provide an older Transformers build; the project uses the
# TrainingArguments API available from 4.45 onward.
transformers_version = Version(version("transformers"))
if transformers_version < Version("4.45.0"):
    print(f"Upgrading Transformers {transformers_version} to the supported 4.45+ range...")
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "--upgrade", "transformers>=4.45.0"
    ])

# PEFT's optional torchao dispatcher errors on Kaggle images with an older
# torchao preinstalled, even when this notebook does not use torchao.
try:
    torchao_version = Version(version("torchao"))
except PackageNotFoundError:
    torchao_version = None
if torchao_version is not None and torchao_version <= Version("0.16.0"):
    print(f"Removing incompatible optional torchao {torchao_version} for PEFT compatibility...")
    subprocess.check_call([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"])
print("Dependencies are installed.")

In [ ]:
import inspect
import json
import shutil
import time
import pandas as pd
import torch
import stanza

from IPython.display import display
from asqp_s2it.data import read_jsonl, category_labels, write_jsonl
from asqp_s2it.metrics import exact_quad_f1_by_instance
from asqp_s2it.model import S2ITPredictor
from asqp_s2it.syntax import DependencyParser
from asqp_s2it.timing import format_duration
from asqp_s2it.training import train

syntax_pipeline = inspect.unwrap(DependencyParser._pipeline)
syntax_source_path = Path(syntax_pipeline.__code__.co_filename)
training_source_path = Path(train.__code__.co_filename)
print(f"Loaded parser source: {syntax_source_path}")
print(f"Loaded training source: {training_source_path}")
if '"lemma": "identity"' not in syntax_source_path.read_text(encoding="utf-8"):
    raise RuntimeError(
        "The Kaggle Input repository is outdated: its parser still requires a language-specific lemma model. "
        "The GitHub clone is stale or has incompatible parser code. Rerun the notebook from the first cell "
        "to pull the latest source, then restart the Kaggle session if the error remains."
    )
if "Preparing {split_name} syntax examples:" not in training_source_path.read_text(encoding="utf-8"):
    raise RuntimeError(
        "The Kaggle Input repository is outdated and has no training-preparation progress logs. "
        "Rerun the notebook from the first cell to pull the latest GitHub source, then restart the Kaggle session if the error remains."
    )

print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 2. Đọc dữ liệu và hiển thị thống kê

Các nhãn lấy từ `quads`: `aspect_term`, `aspect_category`, `opinion_term`, `sentiment`. `null` được giữ nguyên cho aspect/opinion implicit.

In [ ]:
from collections import Counter
from langdetect import DetectorFactory, LangDetectException, detect
from stanza.resources.common import download_resources_json, load_resources_json

DetectorFactory.seed = 0
tagged_data_path = Path(OUTPUT_DIR) / "language_tagged_data"
tagged_data_path.mkdir(parents=True, exist_ok=True)
language_counts = Counter()
prepared_rows = {}
fallback_rows = []
for split_name in ("train", "dev", "test"):
    source_path = data_path / f"{split_name}.jsonl"
    prepared_rows[split_name] = []
    with source_path.open("r", encoding="utf-8") as source_file:
        for line_number, line in enumerate(source_file, start=1):
            if not line.strip():
                continue
            row = json.loads(line)
            if not isinstance(row, dict) or not isinstance(row.get("text"), str):
                raise ValueError(f"{source_path}:{line_number}: expected a JSON object with a text field")
            try:
                language = detect(row["text"]) if row["text"].strip() else None
            except LangDetectException:
                language = None
            if language is None:
                fallback_rows.append((split_name, line_number, row))
            else:
                language_counts[language] += 1
            prepared_rows[split_name].append((line_number, row, language))
if fallback_rows and not language_counts:
    raise ValueError("No record contains language-informative text; add language tags before continuing.")
fallback_language = (
    min(language_counts, key=lambda language: (-language_counts[language], language))
    if language_counts else None
)
if fallback_rows:
    print(
        f"WARNING: assigning fallback language {fallback_language!r} to "
        f"{len(fallback_rows)} empty/undetectable records. Examples: "
        f"{[(split, line) for split, line, _ in fallback_rows[:10]]}"
    )
    for split_name, _, row in fallback_rows:
        row["language"] = fallback_language
        language_counts[fallback_language] += 1
# langdetect recognizes more codes than Stanza has parser models for.
stanza_resources_dir = STANZA_CACHE_DIR if DOWNLOAD_STANZA_MODELS else (STANZA_RESOURCES_INPUT_DIR or STANZA_RESOURCES_DIR)
Path(stanza_resources_dir).mkdir(parents=True, exist_ok=True)
resources_path = Path(stanza_resources_dir) / "resources.json"
if not resources_path.is_file():
    if not DOWNLOAD_STANZA_MODELS:
        raise FileNotFoundError(
            f"Stanza resources.json is required to filter unsupported languages: {resources_path}. "
            "Attach a Stanza resources directory or enable DOWNLOAD_STANZA_MODELS."
        )
    download_resources_json(model_dir=stanza_resources_dir)
stanza_resources = load_resources_json(model_dir=stanza_resources_dir)

def canonical_stanza_language(language):
    resource = stanza_resources.get(language)
    visited = set()
    while isinstance(resource, dict) and resource.get("alias"):
        if language in visited:
            return None
        visited.add(language)
        language = resource["alias"]
        resource = stanza_resources.get(language)
    return language if isinstance(resource, dict) else None

DETECTED_TO_STANZA_LANGUAGE = {
    "en": "en",
    "vi": "vi",
    "zh": "zh-hans",
    "zh-cn": "zh-hans",
    "zh-tw": "zh-hans",
    "zh-hans": "zh-hans",
    "zh-hant": "zh-hans",
    "ja": "ja",
    "de": "de",
    "th": "th",
    "fr": "fr",
}
ALLOWED_STANZA_LANGUAGES = set(DETECTED_TO_STANZA_LANGUAGE.values())

configured_model_language = (
    canonical_stanza_language(STANZA_MODEL_LANGUAGE) if STANZA_MODEL_DIR else None
)
if STANZA_MODEL_DIR and configured_model_language is None:
    raise ValueError(f"Configured Stanza model language is not in resources.json: {STANZA_MODEL_LANGUAGE}")
unsupported_counts = {split_name: Counter() for split_name in prepared_rows}
for split_name, rows in prepared_rows.items():
    retained_rows = []
    for line_number, row, detected_language in rows:
        language = detected_language or fallback_language
        requested_language = DETECTED_TO_STANZA_LANGUAGE.get(language)
        stanza_language = canonical_stanza_language(requested_language) if requested_language else None
        if stanza_language not in ALLOWED_STANZA_LANGUAGES or (
            configured_model_language is not None and stanza_language != configured_model_language
        ):
            unsupported_counts[split_name][language or "unknown"] += 1
            continue
        row["language"] = stanza_language
        retained_rows.append((line_number, row, stanza_language))
    prepared_rows[split_name] = retained_rows
skipped_summary = {
    split_name: dict(counts) for split_name, counts in unsupported_counts.items() if counts
}
if skipped_summary:
    print("Skipped unsupported records (split -> language -> count):", skipped_summary)
if not 0 < DATA_PERCENT <= 100:
    raise ValueError(f"DATA_PERCENT must be in (0, 100], got {DATA_PERCENT}")

import random
sampling_rng = random.Random(SEED)
for split_name, rows in prepared_rows.items():
    if not rows:
        raise ValueError(
            f"No records remain in {split_name} after filtering unsupported Stanza languages; "
            "check language detection or attach matching parser models."
        )
    original_count = len(rows)
    if DATA_PERCENT < 100:
        sample_count = max(1, int(original_count * DATA_PERCENT / 100 + 0.999999))
        selected_indices = sorted(sampling_rng.sample(range(original_count), sample_count))
        rows = [rows[index] for index in selected_indices]
        prepared_rows[split_name] = rows
        print(f"Trial subset {split_name}: {len(rows)} / {original_count} records ({DATA_PERCENT}%)")
    source_path = data_path / f"{split_name}.jsonl"
    destination_path = tagged_data_path / source_path.name
    with destination_path.open("w", encoding="utf-8") as destination_file:
        for line_number, row, language in rows:
            row["language"] = language or fallback_language
            destination_file.write(json.dumps(row, ensure_ascii=False) + "\n")
    print(f"Prepared language-tagged {split_name}: {destination_path}")
language_counts = Counter(
    language for rows in prepared_rows.values() for _, _, language in rows
)
train_records = read_jsonl(tagged_data_path / "train.jsonl")
dev_records = read_jsonl(tagged_data_path / "dev.jsonl")
test_records = read_jsonl(tagged_data_path / "test.jsonl")
PARSER_LANGUAGES = sorted(language_counts)
print("Languages retained for Stanza parsing:", dict(sorted(language_counts.items())))

splits = {"train": train_records, "dev": dev_records, "test": test_records}
split_table = pd.DataFrame([
    {
        "Split": name,
        "Sentences": len(records),
        "Languages": ", ".join(sorted({row.language for row in records})),
        "Quadruples": sum(len(row.quads) for row in records),
        "Implicit aspect": sum(q.aspect_term is None for row in records for q in row.quads),
        "Implicit opinion": sum(q.opinion_term is None for row in records for q in row.quads),
    }
    for name, records in splits.items()
])
display(split_table)

categories = category_labels(train_records)
category_table = pd.DataFrame([
    {"Aspect category": category,
     "Train quadruples": sum(q.aspect_category == category for row in train_records for q in row.quads),
     "Dev quadruples": sum(q.aspect_category == category for row in dev_records for q in row.quads),
     "Test quadruples": sum(q.aspect_category == category for row in test_records for q in row.quads)}
    for category in categories
])
display(category_table)

## 3. Chuẩn bị dependency parser

Notebook chỉ giữ các language đã cho phép: tiếng Anh, Việt, Trung giản thể, Nhật, Đức, Thái và Pháp. Record ngoài danh sách bị loại khỏi từng split và thống kê riêng; do đó Stanza chỉ tải parser cho các language còn lại. Chỉ tải các model `tokenize,pos,depparse` vào `STANZA_CACHE_DIR`, không tải gói `default.zip` đầy đủ. Pipeline dùng lemmatizer identity để đáp ứng prerequisite bắt buộc của depparse mà không cần tải model lemma riêng; cache Hugging Face riêng của từng Stanza language được dọn sau khi tải để giảm dung lượng. Kaggle Internet phải bật.

In [ ]:
parser_setup_started = time.perf_counter()
Path(STANZA_CACHE_DIR).mkdir(parents=True, exist_ok=True)
stanza_model_paths = None
stanza_model_language = None
if DOWNLOAD_STANZA_MODELS:
    STANZA_RESOURCES_DIR = STANZA_CACHE_DIR
elif STANZA_RESOURCES_INPUT_DIR:
    STANZA_RESOURCES_DIR = str(Path(STANZA_RESOURCES_INPUT_DIR).expanduser())
elif STANZA_MODEL_DIR:
    raise ValueError("For online multilingual setup, set STANZA_MODEL_DIR=None and use Stanza resources.")
elif not STANZA_RESOURCES_DIR:
    STANZA_RESOURCES_DIR = STANZA_CACHE_DIR
os.environ["STANZA_RESOURCES_DIR"] = STANZA_RESOURCES_DIR
if DOWNLOAD_STANZA_MODELS:
    for language in PARSER_LANGUAGES:
        print(f"Downloading required Stanza parser processors for {language} ...")
        language_resources_dir = Path(STANZA_RESOURCES_DIR) / language
        if (language_resources_dir / "default.zip").is_file():
            shutil.rmtree(language_resources_dir)
            print(f"Removed previously downloaded full Stanza package for {language}.")
        stanza_hf_cache = Path(HF_CACHE_DIR) / "hub" / f"models--stanfordnlp--stanza-{language}"
        if stanza_hf_cache.exists():
            shutil.rmtree(stanza_hf_cache)
        stanza.download(
            language,
            model_dir=STANZA_RESOURCES_DIR,
            processors="tokenize,pos,depparse",
            verbose=False,
        )
        if stanza_hf_cache.exists():
            shutil.rmtree(stanza_hf_cache)
if not (Path(STANZA_RESOURCES_DIR) / "resources.json").is_file():
    raise FileNotFoundError(f"Stanza resources.json was not created under {STANZA_RESOURCES_DIR}")
parser_setup_seconds = time.perf_counter() - parser_setup_started
dependency_parser = DependencyParser(
    PARSER_LANGUAGE,
    use_gpu=PARSER_USE_GPU,
    model_paths=stanza_model_paths,
    model_language=stanza_model_language,
    resources_dir=STANZA_RESOURCES_DIR,
)
print(f"Parser preparation: {format_duration(parser_setup_seconds)} ({parser_setup_seconds:.2f} s)")

## 4. Huấn luyện S²IT

Ba giai đoạn chạy tuần tự trên cùng LoRA adapter: **(1)** global syntax-guided extraction với tác vụ trích xuất aspect→opinion và opinion→aspect; **(2)** local syntax-guided classification trên các cặp gold aspect–opinion; **(3)** fine-grained structural tuning gồm link prediction theo global dependency relations, node classification và phân loại quadruple theo local one-hop subgraphs. Khi chuẩn bị dữ liệu, log tiến độ mỗi khoảng 5% kèm elapsed/ETA; khi fine-tune, log step/total, %, epoch, loss, elapsed và ETA mỗi 10 steps. Thời gian từng giai đoạn vẫn được ghi riêng và tổng hợp phía dưới.

In [ ]:
if RUN_TRAINING:
    notebook_training_started = time.perf_counter()
    training_metrics = train(
        train_instances=train_records,
        dev_instances=dev_records,
        output_dir=OUTPUT_DIR,
        model_name=BASE_MODEL,
        parser=dependency_parser,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        learning_rate=LEARNING_RATE,
        lora_rank=LORA_RANK,
        max_length=MAX_LENGTH,
    )
    notebook_training_wall_seconds = time.perf_counter() - notebook_training_started
    training_metrics["notebook_training_wall_seconds"] = notebook_training_wall_seconds
    training_metrics["parser_setup_seconds"] = parser_setup_seconds
    training_metrics_path = Path(MODEL_DIR) / "training_metrics.json"
    training_metrics_path.write_text(json.dumps(training_metrics, ensure_ascii=False, indent=2), encoding="utf-8")
else:
    training_metrics_path = Path(MODEL_DIR) / "training_metrics.json"
    if training_metrics_path.is_file():
        training_metrics = json.loads(training_metrics_path.read_text(encoding="utf-8"))
    else:
        training_metrics = None
    print("Training skipped; loading existing adapter if available:", MODEL_DIR)

if training_metrics:
    stage_seconds = training_metrics.get("stage_training_seconds", {})
    timing_rows = [
        {"Phase": "Global syntax-guided extraction", "Seconds": stage_seconds.get("global_extraction", 0.0)},
        {"Phase": "Local syntax-guided classification", "Seconds": stage_seconds.get("local_classification", 0.0)},
        {"Phase": "Fine-grained structural tuning", "Seconds": stage_seconds.get("fine_grained_structure", 0.0)},
        {"Phase": "Total model training", "Seconds": training_metrics.get("training_seconds", 0.0)},
        {"Phase": "End-to-end training (incl. prep/load/save)", "Seconds": training_metrics.get("total_elapsed_seconds", 0.0)},
        {"Phase": "Notebook training call wall time", "Seconds": training_metrics.get("notebook_training_wall_seconds", training_metrics.get("total_elapsed_seconds", 0.0))},
    ]
    timing_table = pd.DataFrame(timing_rows)
    timing_table["Time (HH:MM:SS)"] = timing_table["Seconds"].map(format_duration)
    timing_table["Seconds"] = timing_table["Seconds"].round(2)
    display(timing_table)
    print("Training time:", training_metrics.get("training_time", "n/a"))
    print("End-to-end:", training_metrics.get("total_elapsed_time", "n/a"))
    training_table_path = Path(OUTPUT_DIR) / "training_time.csv"
    timing_table.to_csv(training_table_path, index=False)
    print("Saved:", training_table_path)

## 5. Dự đoán và exact-match F1

Theo phần inference của paper, bước trích xuất giải mã hai chiều aspect→opinion và opinion→aspect bằng beam search kích thước 4; kết quả hai hướng được gộp, sau đó chuyển sang phân loại bằng local one-hop subgraphs. F1 micro chỉ tính quadruple đúng khi cả bốn trường khớp và thuộc đúng instance; không tính partial match. Đặt `MAX_EVAL_RECORDS` thành số nhỏ hơn để chạy thử nhanh.

In [ ]:
if RUN_INFERENCE:
    evaluation_records = test_records if MAX_EVAL_RECORDS is None else test_records[:MAX_EVAL_RECORDS]
    if len(evaluation_records) != len(test_records):
        print(f"Quick evaluation: {len(evaluation_records)} / {len(test_records)} test records")

    predictor = S2ITPredictor(MODEL_DIR, dependency_parser)
    prediction_rows = []
    inference_started = time.perf_counter()
    for index, instance in enumerate(evaluation_records, start=1):
        predicted_quads = predictor.predict(instance, categories, max_new_tokens=MAX_NEW_TOKENS)
        prediction_rows.append({
            "instance_id": instance.instance_id,
            "text": instance.text,
            "language": instance.language,
            "quadruples": predicted_quads,
        })
        if index % 100 == 0 or index == len(evaluation_records):
            print(f"Predicted {index}/{len(evaluation_records)}")
    inference_seconds = time.perf_counter() - inference_started

    prediction_path = Path(OUTPUT_DIR) / "test_predictions.jsonl"
    write_jsonl(prediction_path, prediction_rows)
    gold_by_id = {row.instance_id: row.quads for row in evaluation_records}
    predictions_by_id = {row["instance_id"]: row["quadruples"] for row in prediction_rows}
    evaluation = exact_quad_f1_by_instance(gold_by_id, predictions_by_id)

    result_rows = [
        {"Metric": "Precision", "Value": evaluation["precision"]},
        {"Metric": "Recall", "Value": evaluation["recall"]},
        {"Metric": "Exact quadruple micro-F1", "Value": evaluation["f1"]},
        {"Metric": "True positive quadruples", "Value": evaluation["true_positives"]},
        {"Metric": "Predicted quadruples", "Value": evaluation["predicted"]},
        {"Metric": "Gold quadruples", "Value": evaluation["gold"]},
        {"Metric": "Inference time (HH:MM:SS)", "Value": format_duration(inference_seconds)},
        {"Metric": "Inference time (seconds)", "Value": round(inference_seconds, 2)},
    ]
    if training_metrics:
        result_rows.extend([
            {"Metric": "Training time (HH:MM:SS)", "Value": training_metrics.get("training_time", "n/a")},
            {"Metric": "Training time (seconds)", "Value": round(training_metrics.get("training_seconds", 0.0), 2)},
            {"Metric": "End-to-end training (HH:MM:SS)", "Value": training_metrics.get("total_elapsed_time", "n/a")},
        ])
    results_table = pd.DataFrame(result_rows)
    display(results_table)

    results_csv_path = Path(OUTPUT_DIR) / "evaluation_metrics.csv"
    results_table.to_csv(results_csv_path, index=False)
    print(f"Predictions: {prediction_path}")
    print(f"Metrics CSV: {results_csv_path}")
else:
    print("Inference skipped. Set RUN_INFERENCE=True to predict/evaluate.")

## Các file output

- `s2it_outputs/final/`: LoRA adapter, tokenizer, danh sách category và `training_metrics.json`.
- `s2it_outputs/training_time.csv`: thời gian từng stage và tổng thời gian.
- `s2it_outputs/test_predictions.jsonl`: quadruple dự đoán theo instance.
- `s2it_outputs/evaluation_metrics.csv`: precision, recall, exact quadruple F1 và thời gian inference/training.

Tải các file từ mục **Output** của Kaggle notebook sau khi chạy xong.